In [11]:
"""
Core signal-processing utilities (no pandas/matplotlib).

Includes:
- Robust WAV loading (soundfile → scipy → ffmpeg fallback)
- PSD estimation (periodogram + Welch), spectrogram (Welch-style)
- Simple FM/compliance-ish metrics extraction
- 1D occupancy detection over frequency bins
"""

from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime, timezone
import hashlib
import os
import subprocess
import tempfile
from typing import Dict, List, Optional, Sequence, Tuple, Union

import numpy as np
import torch


# =============================================================================
# Utilities
# =============================================================================

def _int_to_float32(x: np.ndarray) -> np.ndarray:
    """
    Convert integer PCM array to float32 in [-1, 1].

    Args:
        x: Integer numpy array (e.g., int16/int32).

    Returns:
        Float32 numpy array scaled to approximately [-1, 1].

    Raises:
        TypeError: If x is not an integer array.
    """
    if not np.issubdtype(x.dtype, np.integer):
        raise TypeError(f"Expected integer PCM array, got dtype={x.dtype}")
    info = np.iinfo(x.dtype)
    scale = float(max(abs(info.min), abs(info.max)))
    if scale == 0.0:
        return x.astype(np.float32)
    return (x.astype(np.float32) / scale).astype(np.float32, copy=False)


def to_mono_float32(x: np.ndarray) -> np.ndarray:
    """
    Ensure audio is mono float32.

    - Integer PCM → float32 [-1, 1]
    - Multi-channel → average across channels

    Args:
        x: Audio samples as numpy array, shape [T] or [T, C].

    Returns:
        Mono float32 array, shape [T].
    """
    if x.ndim == 0:
        raise ValueError("Audio array is scalar; expected 1D or 2D audio.")
    if np.issubdtype(x.dtype, np.integer):
        x = _int_to_float32(x)
    else:
        x = x.astype(np.float32, copy=False)

    if x.ndim == 2:
        # [T, C] → [T]
        x = x.mean(axis=1, dtype=np.float32)
    elif x.ndim != 1:
        raise ValueError(f"Unsupported audio shape {x.shape}; expected [T] or [T, C].")

    return x.astype(np.float32, copy=False)


def has_soundfile() -> bool:
    """Return True if the `soundfile` package is importable."""
    try:
        import soundfile as _  # noqa: F401
        return True
    except Exception:
        return False


def has_ffmpeg() -> bool:
    """Return True if `ffmpeg` appears to be available on PATH."""
    try:
        return subprocess.call(["ffmpeg", "-version"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL) == 0
    except Exception:
        return False


def load_wav_mono(
    path: Union[str, os.PathLike],
) -> Tuple[torch.Tensor, int]:
    """
    Robust WAV loader that returns a mono float32 torch tensor.

    Load order:
      1) soundfile (handles RF64 and many formats)
      2) scipy.io.wavfile (RIFF/RIFX WAV)
      3) ffmpeg → temporary RIFF WAV → scipy.io.wavfile

    Args:
        path: Path to an audio file (WAV/RF64 or ffmpeg-decodable).

    Returns:
        (waveform, sample_rate)
        waveform: torch.float32 tensor of shape [T], values ~[-1, 1]
        sample_rate: int in Hz

    Raises:
        FileNotFoundError: If file does not exist.
        RuntimeError: If decoding fails and no fallback is available.
        ValueError: If decoded audio is empty.
    """
    path = os.fspath(path)
    if not os.path.isfile(path):
        raise FileNotFoundError(path)

    # 1) soundfile (best, RF64-safe)
    try:
        import soundfile as sf  # type: ignore

        data, sr = sf.read(path, dtype="float32", always_2d=False)
        data = np.asarray(data, dtype=np.float32)
        data = to_mono_float32(data)
        if data.size == 0:
            raise ValueError("Decoded audio is empty.")
        return torch.from_numpy(data).contiguous(), int(sr)
    except Exception:
        pass

    # 2) scipy.io.wavfile
    try:
        from scipy.io import wavfile as scipy_wavfile  # type: ignore

        sr, data = scipy_wavfile.read(path)
        data = to_mono_float32(np.asarray(data))
        if data.size == 0:
            raise ValueError("Decoded audio is empty.")
        return torch.from_numpy(data).contiguous(), int(sr)
    except Exception:
        pass

    # 3) ffmpeg fallback
    if not has_ffmpeg():
        raise RuntimeError(
            "Failed to decode WAV with soundfile/scipy, and ffmpeg is not available.\n"
            "Fix options:\n"
            "  - Install Python package `soundfile` (recommended)\n"
            "  - Or install ffmpeg so conversion can be performed.\n"
            f"Status: soundfile={has_soundfile()}, ffmpeg={has_ffmpeg()}"
        )

    try:
        from scipy.io import wavfile as scipy_wavfile  # type: ignore
    except Exception as e:
        raise RuntimeError("ffmpeg fallback requires scipy (`scipy.io.wavfile`).") from e

    with tempfile.TemporaryDirectory() as td:
        tmp_wav = os.path.join(td, "converted_riff.wav")

        # IMPORTANT: do NOT force -ar (sample rate), preserve original bandwidth.
        cmd = [
            "ffmpeg", "-y",
            "-i", path,
            "-ac", "1",            # mono
            "-c:a", "pcm_s16le",   # standard PCM
            tmp_wav,
        ]
        try:
            subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
        except Exception as e:
            raise RuntimeError(f"ffmpeg conversion failed. Command: {' '.join(cmd)}") from e

        sr, data = scipy_wavfile.read(tmp_wav)
        data = to_mono_float32(np.asarray(data))
        if data.size == 0:
            raise ValueError("Decoded audio is empty.")
        return torch.from_numpy(data).contiguous(), int(sr)


def sha256_file(path: Union[str, os.PathLike], chunk_bytes: int = 1 << 20) -> str:
    """
    Compute SHA-256 hash of a file without loading it entirely into memory.

    Args:
        path: File path.
        chunk_bytes: Read chunk size in bytes.

    Returns:
        Hex digest string.
    """
    h = hashlib.sha256()
    with open(os.fspath(path), "rb") as f:
        while True:
            b = f.read(chunk_bytes)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


# =============================================================================
# Spectral estimation (Torch-first, NumPy only for loading)
# =============================================================================

def _hann_window(n: int, device: torch.device, dtype: torch.dtype) -> torch.Tensor:
    if n <= 0:
        raise ValueError("Window length must be positive.")
    # Periodic=False gives symmetric Hann, typical for spectral analysis
    return torch.hann_window(n, periodic=False, device=device, dtype=dtype)


def periodogram_psd_db(
    waveform: torch.Tensor,
    sample_rate: int,
    *,
    n_fft: Optional[int] = None,
    detrend: bool = True,
    eps: float = 1e-12,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    One-sided periodogram PSD estimate (power spectral density) in dB.

    Args:
        waveform: Audio signal tensor of shape [T] (any device). Will be cast to float32.
        sample_rate: Sampling rate in Hz (>0).
        n_fft: FFT size. If None, uses next power-of-two >= T.
        detrend: If True, subtract mean before FFT.
        eps: Small value to avoid log(0).

    Returns:
        (freq_hz, psd_db)
        freq_hz: [F] frequency axis in Hz (one-sided, includes DC and Nyquist if applicable)
        psd_db:  [F] PSD in dB re (power/Hz)

    Raises:
        ValueError: For empty waveform or invalid parameters.
    """
    if sample_rate <= 0:
        raise ValueError("sample_rate must be > 0")
    x = waveform.reshape(-1).to(dtype=torch.float32)
    if x.numel() == 0:
        raise ValueError("Empty waveform (0 samples).")

    if detrend:
        x = x - x.mean()

    T = int(x.numel())
    if n_fft is None:
        n_fft = 1 << (T - 1).bit_length()
    n_fft = int(n_fft)
    if n_fft <= 0:
        raise ValueError("n_fft must be a positive integer")

    # rFFT (one-sided)
    X = torch.fft.rfft(x, n=n_fft)
    # Rectangular window -> U = sum(w^2) = T (if using length T, but we zero-pad)
    # Use T for normalization of the actual data length.
    psd = (X.real**2 + X.imag**2) / (sample_rate * T)

    # One-sided correction (double non-DC and non-Nyquist bins when n_fft even)
    if n_fft % 2 == 0:
        # bins: 0 ... n_fft/2
        if psd.numel() > 2:
            psd[1:-1] *= 2.0
    else:
        # bins: 0 ... (n_fft-1)/2
        if psd.numel() > 1:
            psd[1:] *= 2.0

    freq = torch.fft.rfftfreq(n_fft, d=1.0 / float(sample_rate)).to(device=x.device)
    psd_db = 10.0 * torch.log10(psd.clamp_min(eps))
    return freq, psd_db


def welch_psd_db(
    waveform: torch.Tensor,
    sample_rate: int,
    *,
    nperseg: int = 2048,
    noverlap: Optional[int] = None,
    n_fft: Optional[int] = None,
    detrend: bool = True,
    average: str = "mean",  # "mean" or "median"
    eps: float = 1e-12,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Welch PSD estimate (one-sided, density power/Hz) in dB.

    Args:
        waveform: Signal tensor [T], cast to float32.
        sample_rate: Sampling rate in Hz (>0).
        nperseg: Segment length in samples (>0).
        noverlap: Overlap length in samples. If None, defaults to nperseg//2.
        n_fft: FFT size (>= nperseg). If None, uses nperseg.
        detrend: If True, remove per-segment mean.
        average: "mean" or "median" across segments.
        eps: Small value to avoid log(0).

    Returns:
        (freq_hz, psd_db)

    Raises:
        ValueError: On invalid parameters or empty input.
    """
    if sample_rate <= 0:
        raise ValueError("sample_rate must be > 0")
    x = waveform.reshape(-1).to(dtype=torch.float32)
    if x.numel() == 0:
        raise ValueError("Empty waveform (0 samples).")

    nperseg = int(nperseg)
    if nperseg <= 0:
        raise ValueError("nperseg must be > 0")

    if noverlap is None:
        noverlap = nperseg // 2
    noverlap = int(noverlap)
    if not (0 <= noverlap < nperseg):
        raise ValueError("noverlap must satisfy 0 <= noverlap < nperseg")

    step = nperseg - noverlap
    if step <= 0:
        raise ValueError("nperseg - noverlap must be > 0")

    if n_fft is None:
        n_fft = nperseg
    n_fft = int(n_fft)
    if n_fft < nperseg:
        raise ValueError("n_fft must be >= nperseg")

    # Pad if too short
    T = int(x.numel())
    if T < nperseg:
        pad = nperseg - T
        x = torch.nn.functional.pad(x, (0, pad))
        T = nperseg

    # Frame using unfold: shape [n_frames, nperseg]
    n_frames = 1 + (T - nperseg) // step
    frames = x.unfold(dimension=0, size=nperseg, step=step)  # [n_frames, nperseg]

    if detrend:
        frames = frames - frames.mean(dim=1, keepdim=True)

    win = _hann_window(nperseg, device=frames.device, dtype=frames.dtype)  # [nperseg]
    U = (win * win).sum().clamp_min(1e-30)  # window power normalization

    frames = frames * win  # broadcast

    X = torch.fft.rfft(frames, n=n_fft, dim=1)
    Pxx = (X.real**2 + X.imag**2) / (float(sample_rate) * U)  # [n_frames, F]

    # One-sided correction
    if n_fft % 2 == 0:
        if Pxx.shape[1] > 2:
            Pxx[:, 1:-1] *= 2.0
    else:
        if Pxx.shape[1] > 1:
            Pxx[:, 1:] *= 2.0

    if average not in ("mean", "median"):
        raise ValueError("average must be 'mean' or 'median'")
    if average == "mean":
        psd = Pxx.mean(dim=0)
    else:
        psd = Pxx.median(dim=0).values

    freq = torch.fft.rfftfreq(n_fft, d=1.0 / float(sample_rate)).to(device=x.device)
    psd_db = 10.0 * torch.log10(psd.clamp_min(eps))
    return freq, psd_db


def welch_spectrogram_db(
    waveform: torch.Tensor,
    sample_rate: int,
    *,
    frame_len: int = 1024,
    hop_len: int = 256,
    n_fft: Optional[int] = None,
    detrend: bool = True,
    eps: float = 1e-12,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Welch-style spectrogram: per-frame one-sided PSD (dB).

    Args:
        waveform: Signal tensor [T], cast to float32.
        sample_rate: Sampling rate in Hz (>0).
        frame_len: Frame length in samples (>0).
        hop_len: Hop length in samples (>0).
        n_fft: FFT size (>= frame_len). If None, uses frame_len.
        detrend: If True, remove per-frame mean.
        eps: Small value to avoid log(0).

    Returns:
        (time_s, freq_hz, spec_db)
        time_s:  [n_frames] frame start times in seconds
        freq_hz: [F] one-sided frequency axis
        spec_db: [n_frames, F] PSD in dB re (power/Hz)

    Raises:
        ValueError: On invalid parameters or empty input.
    """
    if sample_rate <= 0:
        raise ValueError("sample_rate must be > 0")
    x = waveform.reshape(-1).to(dtype=torch.float32)
    if x.numel() == 0:
        raise ValueError("Empty waveform (0 samples).")

    frame_len = int(frame_len)
    hop_len = int(hop_len)
    if frame_len <= 0 or hop_len <= 0:
        raise ValueError("frame_len and hop_len must be positive integers")

    if n_fft is None:
        n_fft = frame_len
    n_fft = int(n_fft)
    if n_fft < frame_len:
        raise ValueError("n_fft must be >= frame_len")

    T = int(x.numel())
    if T < frame_len:
        x = torch.nn.functional.pad(x, (0, frame_len - T))
        T = frame_len

    n_frames = 1 + (T - frame_len) // hop_len
    frames = x.unfold(0, frame_len, hop_len)  # [n_frames, frame_len]

    if detrend:
        frames = frames - frames.mean(dim=1, keepdim=True)

    win = _hann_window(frame_len, device=frames.device, dtype=frames.dtype)
    U = (win * win).sum().clamp_min(1e-30)
    frames = frames * win

    X = torch.fft.rfft(frames, n=n_fft, dim=1)
    Pxx = (X.real**2 + X.imag**2) / (float(sample_rate) * U)

    # One-sided correction
    if n_fft % 2 == 0:
        if Pxx.shape[1] > 2:
            Pxx[:, 1:-1] *= 2.0
    else:
        if Pxx.shape[1] > 1:
            Pxx[:, 1:] *= 2.0

    spec_db = 10.0 * torch.log10(Pxx.clamp_min(eps))
    freq = torch.fft.rfftfreq(n_fft, d=1.0 / float(sample_rate)).to(device=x.device)
    time_s = (torch.arange(n_frames, device=x.device, dtype=torch.float32) * float(hop_len)) / float(sample_rate)
    return time_s, freq, spec_db


# =============================================================================
# Metrics / occupancy
# =============================================================================

@dataclass(frozen=True)
class OccupiedBand:
    """A contiguous occupied frequency region."""
    f_start_hz: float
    f_stop_hz: float
    peak_freq_hz: float
    peak_power_db: float


def estimate_noise_floor_db(psd_db: torch.Tensor) -> torch.Tensor:
    """
    Robust noise-floor proxy: median of the lower 50% of PSD bins.

    Args:
        psd_db: PSD in dB, shape [F].

    Returns:
        Scalar tensor (dB).
    """
    v = psd_db.reshape(-1)
    k = max(1, v.numel() // 2)
    lower = torch.sort(v).values[:k]
    return lower.median()


def detect_occupancy_1d(
    psd_db: torch.Tensor,
    freq_hz: torch.Tensor,
    *,
    margin_db: float = 6.0,
    min_bins: int = 3,
) -> Tuple[torch.Tensor, torch.Tensor, List[OccupiedBand]]:
    """
    Binary occupancy detection over frequency bins + contiguous band extraction.

    Args:
        psd_db: PSD in dB, shape [F].
        freq_hz: Frequency axis in Hz, shape [F] (monotone increasing).
        margin_db: Threshold above noise floor to declare occupancy.
        min_bins: Minimum contiguous bins to keep a detected band.

    Returns:
        (occupancy, noise_floor_db, bands)
        occupancy: [F] float tensor in {0,1}
        noise_floor_db: scalar tensor (dB)
        bands: list of OccupiedBand for contiguous occupied regions

    Raises:
        ValueError: If shapes mismatch or too short.
    """
    psd_db = psd_db.reshape(-1)
    freq_hz = freq_hz.reshape(-1)
    if psd_db.numel() != freq_hz.numel():
        raise ValueError(f"psd_db and freq_hz must have same length, got {psd_db.numel()} vs {freq_hz.numel()}")
    if psd_db.numel() < 2:
        raise ValueError("Need at least 2 frequency bins.")

    noise_floor = estimate_noise_floor_db(psd_db)
    occ = (psd_db > (noise_floor + float(margin_db))).to(dtype=torch.float32)

    bands: List[OccupiedBand] = []
    if occ.sum().item() == 0:
        return occ, noise_floor, bands

    # Find contiguous runs where occ == 1
    occ_bool = occ.to(dtype=torch.bool)
    idx = torch.nonzero(occ_bool, as_tuple=False).reshape(-1)
    # contiguous split
    splits = torch.where(idx[1:] != idx[:-1] + 1)[0] + 1
    runs = torch.split(idx, splits.tolist())

    for run in runs:
        if run.numel() < int(min_bins):
            continue
        i0 = int(run[0].item())
        i1 = int(run[-1].item())

        # Peak within run
        run_psd = psd_db[run]
        peak_rel = int(torch.argmax(run_psd).item())
        peak_i = int(run[peak_rel].item())

        bands.append(
            OccupiedBand(
                f_start_hz=float(freq_hz[i0].item()),
                f_stop_hz=float(freq_hz[i1].item()),
                peak_freq_hz=float(freq_hz[peak_i].item()),
                peak_power_db=float(psd_db[peak_i].item()),
            )
        )

    return occ, noise_floor, bands


def extract_fm_metrics(
    psd_db: torch.Tensor,
    freq_hz: torch.Tensor,
    *,
    sample_rate: int,
    pilot_hz: float = 19_000.0,
    pilot_margin_db: float = 6.0,
) -> Dict[str, Union[float, int, bool, str]]:
    """
    Lightweight metrics useful for FM monitoring/compliance-style checks.

    Metrics:
      - noise floor proxy (dB)
      - peak frequency / peak power
      - SNR proxy = peak - noise_floor
      - 19 kHz pilot detection (if within Nyquist)

    Args:
        psd_db: PSD in dB, shape [F].
        freq_hz: Frequency axis in Hz, shape [F].
        sample_rate: Sampling rate in Hz (>0).
        pilot_hz: Pilot tone frequency to test (default 19 kHz).
        pilot_margin_db: Pilot detection threshold above noise floor.

    Returns:
        Dict with scalar metrics.

    Raises:
        ValueError: If shapes mismatch or invalid sample_rate.
    """
    if sample_rate <= 0:
        raise ValueError("sample_rate must be > 0")
    psd_db = psd_db.reshape(-1)
    freq_hz = freq_hz.reshape(-1)
    if psd_db.numel() != freq_hz.numel():
        raise ValueError("psd_db and freq_hz must have same length")

    noise_floor = estimate_noise_floor_db(psd_db)
    peak_i = int(torch.argmax(psd_db).item())
    peak_freq = float(freq_hz[peak_i].item())
    peak_power = float(psd_db[peak_i].item())
    snr_proxy = float((psd_db[peak_i] - noise_floor).item())

    pilot_detected = False
    if pilot_hz <= (sample_rate / 2.0) and psd_db.numel() > 1:
        pilot_i = int(torch.argmin(torch.abs(freq_hz - float(pilot_hz))).item())
        pilot_detected = bool(psd_db[pilot_i] > (noise_floor + float(pilot_margin_db)))

    return {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "sample_rate_hz": int(sample_rate),
        "noise_floor_db": float(noise_floor.item()),
        "peak_frequency_hz": peak_freq,
        "peak_power_db": peak_power,
        "snr_proxy_db": snr_proxy,
        "pilot_19khz_detected": pilot_detected,
    }


def generate_audit_record(
    wav_path: Union[str, os.PathLike],
    *,
    sample_rate: int,
    duration_s: float,
    metrics: Dict[str, Union[float, int, bool, str]],
) -> Dict[str, Union[str, int, float, bool]]:
    """
    Evidence packaging: attach file hash + key metrics.

    Args:
        wav_path: Input file path.
        sample_rate: Sampling rate in Hz.
        duration_s: Duration in seconds.
        metrics: Dict from `extract_fm_metrics` (or similar).

    Returns:
        Dictionary suitable for JSON serialization.
    """
    wav_path = os.fspath(wav_path)
    return {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "file_path": wav_path,
        "file_hash_sha256": sha256_file(wav_path),
        "sample_rate_hz": int(sample_rate),
        "duration_s": float(duration_s),
        "peak_frequency_hz": float(metrics.get("peak_frequency_hz", 0.0)),  # type: ignore[arg-type]
        "snr_proxy_db": float(metrics.get("snr_proxy_db", 0.0)),            # type: ignore[arg-type]
        "pilot_19khz_detected": bool(metrics.get("pilot_19khz_detected", False)),  # type: ignore[arg-type]
    }


# =============================================================================
# Minimal pipeline helper (kept separate from visualization)
# =============================================================================

@torch.no_grad()
def analyze_wav_basic(
    wav_path: Union[str, os.PathLike],
    *,
    welch_nperseg: int = 2048,
    welch_noverlap: Optional[int] = None,
    welch_nfft: Optional[int] = None,
    occ_margin_db: float = 6.0,
) -> Dict[str, object]:
    """
    End-to-end “core analysis” without plotting:
      load → Welch PSD → metrics → occupancy bands → audit record.

    Args:
        wav_path: Path to WAV/RF64 (or ffmpeg-decodable).
        welch_nperseg/noverlap/nfft: Welch parameters.
        occ_margin_db: Occupancy threshold margin above noise floor.

    Returns:
        Dict containing waveform info, PSD, metrics, bands, and audit record.
    """
    x, sr = load_wav_mono(wav_path)
    duration_s = float(x.numel()) / float(sr)

    f, psd_db = welch_psd_db(
        x, sr,
        nperseg=welch_nperseg,
        noverlap=welch_noverlap,
        n_fft=welch_nfft,
        detrend=True,
        average="mean",
    )

    metrics = extract_fm_metrics(psd_db, f, sample_rate=sr)
    occ, noise_floor_db, bands = detect_occupancy_1d(psd_db, f, margin_db=occ_margin_db)

    audit = generate_audit_record(
        wav_path,
        sample_rate=sr,
        duration_s=duration_s,
        metrics=metrics,
    )

    return {
        "sample_rate_hz": sr,
        "duration_s": duration_s,
        "freq_hz": f,
        "psd_db": psd_db,
        "noise_floor_db": noise_floor_db,
        "occupancy": occ,
        "bands": bands,
        "metrics": metrics,
        "audit_record": audit,
    }

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=294a8efe-915d-440c-bc69-9df4f4b79046' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>

In [12]:
from datasets import load_dataset
from IPython.display import Audio, display

# Load one voice sample from a stable public speech dataset.
voice_dataset = load_dataset(
    "librispeech_asr",
    "clean",
    split="validation",
    streaming=True,
)
voice_sample = next(iter(voice_dataset))

audio = voice_sample["audio"]
transcript = voice_sample["text"]

display(Audio(audio["array"], rate=audio["sampling_rate"]))
print(transcript)


ImportError: To support decoding audio data, please install 'torchcodec'.